In [1]:
import pandas as pd
import numpy as np

In [2]:
df_factor_monthly = pd.read_parquet('df_factor_monthly.parquet')
df_factor_monthly

,Symbol,YearMonth,TradingDate,is_hs300,factor_mom_2d,factor_mom_5d,factor_mom_10d,factor_mom_20d,factor_reversal_5d,factor_pos_10d,...,factor_br,factor_avg_px_bias,factor_up_vol_ratio,factor_daily_pos,factor_pvt_20d,factor_turnover_bias,factor_pv_divergence,target_monthly_ret_x,is_month_end,target_monthly_ret_y
0,600779,2010-02,2010-02-12,True,0.690121,0.848414,1.612065,-0.248941,-0.848414,1.676608,...,-0.673930,-1.113377,-0.645188,-0.186050,-0.503809,1.344352,0.0,-0.008841,True,-0.089385
1,600029,2010-02,2010-02-22,True,-0.588621,-0.219344,0.438071,0.139179,0.219344,0.855814,...,-0.040977,-0.508716,0.708936,-0.039847,1.125171,0.568138,0.0,0.158887,True,0.094646
2,000413,2010-02,2010-02-25,False,-1.672973,-1.137782,-1.338224,-1.119934,1.137782,-1.283195,...,-1.526219,0.911112,0.590471,-0.429389,0.052383,-0.549047,0.0,0.049131,True,0.032340
3,000553,2010-02,2010-02-25,False,0.309649,0.176796,-0.991135,3.429432,-0.176796,-1.712070,...,2.455148,-0.231638,2.660082,-0.618859,2.123024,-0.915525,0.0,0.062428,True,0.066126
4,000651,2010-02,2010-02-25,True,-0.119526,0.123277,-0.101755,0.180504,-0.123277,0.331209,...,1.034304,-0.409951,0.578066,-0.506111,0.039669,1.663298,0.0,0.081640,True,0.098067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115653,688472,2024-11,2024-11-29,False,0.942669,2.994179,1.146609,0.034508,-2.994179,2.431883,...,0.348515,2.346973,0.651500,2.662617,-1.095306,0.622793,0.0,-0.157664,True,-0.157346
115654,688506,2024-11,2024-11-29,False,2.290105,2.503464,2.086610,2.415068,-2.503464,1.941901,...,1.661631,2.886909,1.816233,-0.448256,-0.170427,2.464259,0.0,-0.114185,True,-0.199190
115655,688561,2024-11,2024-11-29,False,1.600082,-1.179198,-0.612583,-0.978070,1.179198,-0.078770,...,-0.216474,1.835178,-0.574994,0.132964,1.115176,0.078435,0.0,-0.078899,True,-0.133112
115656,688599,2024-11,2024-11-29,True,0.560486,0.825428,0.879359,0.578973,-0.825428,1.074836,...,0.152875,2.215543,0.423418,0.037307,0.108796,-0.895662,0.0,-0.195804,True,-0.252351


In [3]:
import pandas as pd
import numpy as np
import cvxpy as cp
from sklearn.linear_model import Ridge
from sklearn.covariance import LedoitWolf
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import scipy.stats as stats
import warnings
import gc

warnings.filterwarnings("ignore")

def numpy_spearmanr(x, y):
    if len(x) < 2: return 0.0
    res = stats.spearmanr(x, y)
    return res.correlation if hasattr(res, 'correlation') else res[0]

# --- 1. 配置 ---
df = df_factor_monthly.copy()
target = 'target_monthly_ret_y' 
features = [col for col in df.columns if col.startswith('factor_')]

oos_start, oos_end = '2013-06-30', '2024-12-31'
lookback_window = 60    # 使用 5 年回溯窗口
train_step = 6         # 半年更新一次 Alpha
risk_window_len = 36    

df['TradingDate'] = pd.to_datetime(df['TradingDate'])
all_dates = sorted(df['TradingDate'].unique())
rebalance_dates = [d for d in all_dates if pd.to_datetime(oos_start) <= d <= pd.to_datetime(oos_end)]
df_lookup = {d: group for d, group in df.groupby('TradingDate')}

# --- 2. 状态存储 ---
weight_records, performance_log = [], [] 
oos_num_raw, oos_denominator_list = [], [] 
last_w_val, model = None, None

# 固定 Alpha 参数：针对 84 个原始因子，5000 提供较强的鲁棒性
FIXED_ALPHA = 5000

print(f"🚀 启动 Ridge-MVO 策略 (固定 Alpha={FIXED_ALPHA} + 原始特征)...")

# --- 3. 滚动预测与优化 ---
for i in tqdm(range(len(rebalance_dates))):
    date = rebalance_dates[i]
    date_idx = all_dates.index(date)
    if date_idx < lookback_window: continue
    
    # --- 3.1 动态再训练 (固定 Alpha 逻辑) ---
    if i % train_step == 0 or model is None:
        gc.collect()
        train_dates = all_dates[date_idx - lookback_window : date_idx]
        # 获取训练集并清洗
        full_train = pd.concat([df_lookup[d] for d in train_dates if d in df_lookup])[[target] + features].dropna()
        
        # 使用固定 Alpha 在全量窗口训练，无需再进行 train_test_split 搜索
        model = Ridge(alpha=FIXED_ALPHA)
        model.fit(full_train[features], full_train[target])

    # --- 3.2 样本外预测 ---
    current_day = df_lookup.get(date)
    if current_day is None: continue
    current_day = current_day.dropna(subset=features).copy()
    
    mu_raw_output = model.predict(current_day[features])
    mu_series = pd.Series(mu_raw_output, index=current_day['Symbol'].tolist())

    # --- 3.3 MVO 配置 ---
    trade_candidates = current_day[current_day['is_hs300'] == True]['Symbol'].tolist()
    risk_dates = all_dates[date_idx - risk_window_len : date_idx]
    pm_dates = all_dates[max(0, date_idx - 60) : date_idx]
    
    risk_hist_df = pd.concat([df_lookup[d][['TradingDate', 'Symbol', target]] for d in risk_dates if d in df_lookup])
    pm_hist_df = pd.concat([df_lookup[d][['Symbol', target]] for d in pm_dates if d in df_lookup])
    rolling_pm_series = pm_hist_df.groupby('Symbol')[target].mean()
    
    v_matrix = risk_hist_df[risk_hist_df['Symbol'].isin(trade_candidates)].pivot(index='TradingDate', columns='Symbol', values=target).fillna(0)
    active_symbols = [s for s in trade_candidates if s in v_matrix.columns]
    
    if len(active_symbols) < 50: continue 

    mu_for_mvo = mu_series.reindex(active_symbols).fillna(0).values
    Sigma = LedoitWolf().fit(v_matrix[active_symbols].values).covariance_
    Sigma = (Sigma + Sigma.T) / 2
    
    n = len(active_symbols)
    w = cp.Variable(n)
    curr_last_w = last_w_val.reindex(active_symbols).fillna(0).values if last_w_val is not None else np.ones(n)/n
    
    # 目标函数：最大化预期收益 - 风险惩罚 - 换手率惩罚
    obj = cp.Maximize(mu_for_mvo @ w - 0.5 * 5.0 * cp.quad_form(w, Sigma) - 0.03 * cp.norm(w - curr_last_w, 1))
    constraints = [cp.sum(w) == 1, w >= 0, w <= 1.0] 
    
    try:
        prob = cp.Problem(obj, constraints)
        prob.solve(solver=cp.OSQP)
        weights = np.array(w.value).flatten()
        if weights is None: raise ValueError()
        weights[weights < 1e-4] = 0
        weights /= weights.sum()
    except:
        weights = curr_last_w
        
    last_w_val = pd.Series(weights, index=active_symbols)
    weight_records.append(pd.DataFrame({'date': date, 'symbol': active_symbols, 'weight': weights}))
    
    # --- 3.4 统计诊断 ---
    if target in current_day.columns:
        y_real = current_day.set_index('Symbol').reindex(active_symbols)[target].values
        daily_pm_bench = rolling_pm_series.reindex(active_symbols).fillna(0).values

        err_sq_raw = np.sum((y_real - mu_for_mvo)**2)
        ic_raw = numpy_spearmanr(mu_for_mvo, y_real)
        bench_err_sq = np.sum((y_real - daily_pm_bench)**2)

        oos_num_raw.append(err_sq_raw)
        oos_denominator_list.append(bench_err_sq)

        performance_log.append({
            'Date': date, 'IC_Raw': ic_raw,
            'R2_OS_Raw': 1 - (err_sq_raw / (bench_err_sq + 1e-8)),
            'Turnover': np.sum(np.abs(weights - curr_last_w)),
            'Alpha_Used': model.alpha
        })

# --- 4. 导出 ---
if weight_records:
    total_bench = sum(oos_denominator_list) + 1e-8
    final_r2_raw = 1 - (sum(oos_num_raw) / total_bench)
    perf_df = pd.DataFrame(performance_log)
    print(f"\n✅ Ridge 汇总 R2_OS: {final_r2_raw:.6f} | 平均 IC: {perf_df['IC_Raw'].mean():.4f}")
    
    # 输出指定名称的 Excel
    perf_df.to_excel('Ridge_MVO_Monthly_Diagnostics.xlsx', index=False)
    pd.concat(weight_records)[lambda x: x['weight'] > 0].to_excel('Ridge_MVO_Monthly_Weights.xlsx', index=False)

🚀 启动 Ridge-MVO 策略 (固定 Alpha=5000 + 原始特征)...


100%|██████████| 514/514 [01:19<00:00,  6.47it/s]



✅ Ridge 汇总 R2_OS: 0.043662 | 平均 IC: 0.0368


In [4]:
import pandas as pd
import numpy as np
import cvxpy as cp
import xgboost as xgb
from sklearn.covariance import LedoitWolf
from tqdm import tqdm
from scipy.stats import spearmanr
import warnings
import gc

warnings.filterwarnings("ignore")

# --- 0. 稳健辅助函数 ---
def numpy_spearmanr(x, y):
    if len(x) < 2 or len(y) < 2: return 0.0
    res = spearmanr(x, y)
    return res.correlation if hasattr(res, 'correlation') else res[0]

# --- 1. 配置与数据准备 ---
df = df_factor_monthly.copy()
target = 'target_monthly_ret_y' 
features = [col for col in df.columns if col.startswith('factor_')]

oos_start, oos_end = '2013-06-30', '2024-12-31'
lookback_window = 60    # 5年回溯
train_step = 6         # 半年更新一次
risk_window_len = 36   

# 固定 XGBoost 参数：针对 84 个因子设计的稳健参数组合
FIXED_XGB_PARAMS = {
    'n_estimators': 80,           # 适度减少树的数量，防止在长窗口末端过拟合
    'max_depth': 2,               # 【关键】从 3 降到 2。深度为 2 的树只能捕捉二阶交互，更稳健
    'learning_rate': 0.005,       # 进一步降低学习率，让模型收敛更慢、更平滑
    'subsample': 0.7,             # 引入行采样，增加基学习器之间的差异性
    'colsample_bytree': 0.6,      # 降低列采样比例，强制模型在训练每棵树时只看部分因子
    'reg_lambda': 500.0,          # 【大幅提升】L2 正则化从 100 提到 500，强行压低叶子节点权重
    'reg_alpha': 10.0,            # 引入 L1 正则化，让一些贡献极小的无效因子权重彻底归零
    'objective': 'reg:squarederror', # 尝试换回平方损失，配合高正则化，通常对解释 $R^2$ 更客观
    'tree_method': 'hist',
    'n_jobs': -1,
    'random_state': 2026,
    'verbosity': 0
}

df['TradingDate'] = pd.to_datetime(df['TradingDate'])
all_dates = sorted(df['TradingDate'].unique())
rebalance_dates = [d for d in all_dates if pd.to_datetime(oos_start) <= d <= pd.to_datetime(oos_end)]
df_lookup = {d: group for d, group in df.groupby('TradingDate')}

# --- 3. 状态存储变量 ---
weight_records, performance_log = [], [] 
oos_num_raw, oos_denominator_list = [], [] 
last_w_val, model = None, None

print(f"🚀 启动 MAE-XGBoost 滚动回测 (固定参数 + 全量训练 + 单股上限 1.0)...")

# --- 4. 滚动预测与优化循环 ---
for i in tqdm(range(len(rebalance_dates))):
    date = rebalance_dates[i]
    date_idx = all_dates.index(date)
    if date_idx < lookback_window: continue
    
    # --- 4.1 动态再训练 (固定参数逻辑) ---
    if i % train_step == 0 or model is None:
        gc.collect()
        train_dates = all_dates[date_idx - lookback_window : date_idx]
        # 使用全量窗口数据，不再拆分验证集
        full_train = pd.concat([df_lookup[d] for d in train_dates if d in df_lookup])[[target] + features].dropna()
        
        # 实例化固定参数模型
        model = xgb.XGBRegressor(**FIXED_XGB_PARAMS)
        
        # 全量拟合
        model.fit(full_train[features], full_train[target])

    # --- 4.2 样本外预测 ---
    current_day = df_lookup.get(date)
    if current_day is None: continue
    current_day = current_day.dropna(subset=features).copy()
    
    mu_raw_output = model.predict(current_day[features])
    mu_series = pd.Series(mu_raw_output, index=current_day['Symbol'].tolist())

    # --- 4.3 MVO 资产配置 (逻辑同 Ridge 模型) ---
    trade_candidates = current_day[current_day['is_hs300'] == True]['Symbol'].tolist()
    risk_dates = all_dates[date_idx - risk_window_len : date_idx]
    pm_dates = all_dates[max(0, date_idx - 60) : date_idx]
    
    risk_hist_df = pd.concat([df_lookup[d][['TradingDate', 'Symbol', target]] for d in risk_dates if d in df_lookup])
    pm_hist_df = pd.concat([df_lookup[d][['Symbol', target]] for d in pm_dates if d in df_lookup])
    rolling_pm_series = pm_hist_df.groupby('Symbol')[target].mean()
    
    v_matrix = risk_hist_df[risk_hist_df['Symbol'].isin(trade_candidates)].pivot(index='TradingDate', columns='Symbol', values=target).fillna(0)
    active_symbols = [s for s in trade_candidates if s in v_matrix.columns]
    
    if len(active_symbols) < 50: continue 

    mu_for_mvo = mu_series.reindex(active_symbols).fillna(0).values
    Sigma = LedoitWolf().fit(v_matrix[active_symbols].values).covariance_
    Sigma = (Sigma + Sigma.T) / 2
    
    n = len(active_symbols)
    w = cp.Variable(n)
    curr_last_w = last_w_val.reindex(active_symbols).fillna(0).values if last_w_val is not None else np.ones(n)/n
    
    # 目标函数：最大化预期收益 - 风险惩罚 - 换手惩罚
    obj = cp.Maximize(mu_for_mvo @ w - 0.5 * 5.0 * cp.quad_form(w, Sigma) - 0.03 * cp.norm(w - curr_last_w, 1))
    constraints = [cp.sum(w) == 1, w >= 0, w <= 1.0] 
    
    try:
        prob = cp.Problem(obj, constraints)
        prob.solve(solver=cp.OSQP, eps_abs=1e-5, eps_rel=1e-5)
        weights = np.array(w.value).flatten()
        if weights is None: raise ValueError()
        weights[weights < 1e-4] = 0; weights /= weights.sum()
    except:
        weights = curr_last_w
        
    last_w_val = pd.Series(weights, index=active_symbols)
    weight_records.append(pd.DataFrame({'date': date, 'symbol': active_symbols, 'weight': weights}))
    
    # --- 4.4 统计诊断 ---
    if target in current_day.columns:
        y_real = current_day.set_index('Symbol').reindex(active_symbols)[target].values
        daily_pm_bench = rolling_pm_series.reindex(active_symbols).fillna(0).values

        err_sq_raw = np.sum((y_real - mu_for_mvo)**2)
        ic_raw = numpy_spearmanr(mu_for_mvo, y_real)
        bench_err_sq = np.sum((y_real - daily_pm_bench)**2)

        oos_num_raw.append(err_sq_raw)
        oos_denominator_list.append(bench_err_sq)

        performance_log.append({
            'Date': date, 'IC_Raw': ic_raw,
            'R2_OS_Raw': 1 - (err_sq_raw / (bench_err_sq + 1e-8)),
            'Turnover': np.sum(np.abs(weights - curr_last_w))
        })

# --- 5. 汇总导出 ---
if weight_records:
    total_bench = sum(oos_denominator_list) + 1e-8
    final_r2_raw = 1 - (sum(oos_num_raw) / total_bench)
    perf_df = pd.DataFrame(performance_log)
    
    print("\n" + "="*45)
    print(f"✅ XGBoost 汇总 R2_OS: {final_r2_raw:.6f} | 平均 IC: {perf_df['IC_Raw'].mean():.4f}")
    
    # 导出指定文件
    perf_df.to_excel('XGB_Monthly_Diagnostics.xlsx', index=False)
    pd.concat(weight_records)[lambda x: x['weight'] > 0].to_excel('XGB_Monthly_Weights.xlsx', index=False)

🚀 启动 MAE-XGBoost 滚动回测 (固定参数 + 全量训练 + 单股上限 1.0)...


100%|██████████| 514/514 [02:22<00:00,  3.61it/s]



✅ XGBoost 汇总 R2_OS: 0.051486 | 平均 IC: 0.0177


In [5]:
import pandas as pd
import numpy as np
import cvxpy as cp
import lightgbm as lgb
from sklearn.covariance import LedoitWolf
from tqdm import tqdm
import scipy.stats as stats
import warnings
import gc

warnings.filterwarnings("ignore")

def numpy_spearmanr(x, y):
    if len(x) < 2: return 0.0
    res = stats.spearmanr(x, y)
    return res.correlation if hasattr(res, 'correlation') else res[0]

# --- 1. 配置 ---
df = df_factor_monthly.copy()
target = 'target_monthly_ret_y' 
features = [col for col in df.columns if col.startswith('factor_')]

oos_start, oos_end = '2013-06-30', '2024-12-31'
lookback_window = 60    # 5年回溯
train_step = 6         # 半年更新一次
risk_window_len = 36    

# 固定 LightGBM 参数：移除早停逻辑，设定固定迭代次数
FIXED_LGBM_PARAMS = {
    'n_estimators': 80,           # 降低迭代次数
    'learning_rate': 0.005,       # 极致低学习率，增加收敛稳定性
    'max_depth': 2,               # 【核心】降到 2 层。强制捕捉最简单的逻辑
    'num_leaves': 3,              # 配合 max_depth=2，叶子数只能是 3 或 4
    'subsample': 0.7,             # 样本采样，增加鲁棒性
    'subsample_freq': 1,
    'colsample_bytree': 0.6,      # 特征采样，防止模型依赖某几个特定因子
    'lambda_l1': 10.0,            # 增加 L1 正则，让模型自动剔除弱因子
    'lambda_l2': 500.0,           # 【大幅提升】L2 正则，压低系数绝对值
    'min_child_samples': 50,      # 强制每个叶子节点必须有更多样本，防止对个别股票过拟合
    'objective': 'huber',         # 维持 Huber，对抗 A 股收益率的厚尾噪声
    'verbosity': -1,
    'random_state': 2026,
    'n_jobs': -1
}

df['TradingDate'] = pd.to_datetime(df['TradingDate'])
all_dates = sorted(df['TradingDate'].unique())
rebalance_dates = [d for d in all_dates if pd.to_datetime(oos_start) <= d <= pd.to_datetime(oos_end)]
df_lookup = {d: group for d, group in df.groupby('TradingDate')}

# --- 2. 状态存储 ---
weight_records, performance_log = [], [] 
oos_num_raw, oos_denominator_list = [], [] 
last_w_val, model = None, None

print(f"🚀 启动 Huber-LGBM 滚动回测 (固定参数 + 全量训练 + 单股上限 1.0)...")

# --- 3. 滚动预测与优化 ---
for i in tqdm(range(len(rebalance_dates))):
    date = rebalance_dates[i]
    date_idx = all_dates.index(date)
    if date_idx < lookback_window: continue
    
    # --- 3.1 动态再训练 (固定参数逻辑) ---
    if i % train_step == 0 or model is None:
        gc.collect()
        train_dates = all_dates[date_idx - lookback_window : date_idx]
        # 使用全量窗口数据训练，不再进行验证集拆分
        full_train = pd.concat([df_lookup[d] for d in train_dates if d in df_lookup])[[target] + features].dropna()
        
        # 实例化固定参数模型
        model = lgb.LGBMRegressor(**FIXED_LGBM_PARAMS)
        
        # 全量拟合，移除 callbacks 和 eval_set
        model.fit(full_train[features], full_train[target])

    # --- 3.2 预测 ---
    current_day = df_lookup.get(date)
    if current_day is None: continue
    current_day = current_day.dropna(subset=features).copy()
    
    mu_raw_output = model.predict(current_day[features])
    mu_series = pd.Series(mu_raw_output, index=current_day['Symbol'].tolist())

    # --- 3.3 MVO 配置 (逻辑与前述模型保持完全一致) ---
    trade_candidates = current_day[current_day['is_hs300'] == True]['Symbol'].tolist()
    risk_dates = all_dates[date_idx - risk_window_len : date_idx]
    pm_dates = all_dates[max(0, date_idx - 60) : date_idx]
    
    risk_hist_df = pd.concat([df_lookup[d][['TradingDate', 'Symbol', target]] for d in risk_dates if d in df_lookup])
    pm_hist_df = pd.concat([df_lookup[d][['Symbol', target]] for d in pm_dates if d in df_lookup])
    rolling_pm_series = pm_hist_df.groupby('Symbol')[target].mean()
    
    v_matrix = risk_hist_df[risk_hist_df['Symbol'].isin(trade_candidates)].pivot(index='TradingDate', columns='Symbol', values=target).fillna(0)
    active_symbols = [s for s in trade_candidates if s in v_matrix.columns]
    
    if len(active_symbols) < 50: continue 

    mu_for_mvo = mu_series.reindex(active_symbols).fillna(0).values
    Sigma = LedoitWolf().fit(v_matrix[active_symbols].values).covariance_
    Sigma = (Sigma + Sigma.T) / 2
    
    n = len(active_symbols)
    w = cp.Variable(n)
    curr_last_w = last_w_val.reindex(active_symbols).fillna(0).values if last_w_val is not None else np.ones(n)/n
    
    # 目标函数
    obj = cp.Maximize(mu_for_mvo @ w - 0.5 * 5.0 * cp.quad_form(w, Sigma) - 0.03 * cp.norm(w - curr_last_w, 1))
    constraints = [cp.sum(w) == 1, w >= 0, w <= 1.0] 
    
    try:
        prob = cp.Problem(obj, constraints)
        prob.solve(solver=cp.OSQP)
        weights = np.array(w.value).flatten()
        if weights is None: raise ValueError()
        weights[weights < 1e-4] = 0; weights /= weights.sum()
    except:
        weights = curr_last_w
        
    last_w_val = pd.Series(weights, index=active_symbols)
    weight_records.append(pd.DataFrame({'date': date, 'symbol': active_symbols, 'weight': weights}))
    
    # --- 3.4 统计诊断 ---
    if target in current_day.columns:
        y_real = current_day.set_index('Symbol').reindex(active_symbols)[target].values
        daily_pm_bench = rolling_pm_series.reindex(active_symbols).fillna(0).values

        err_sq_raw = np.sum((y_real - mu_for_mvo)**2)
        ic_raw = numpy_spearmanr(mu_for_mvo, y_real)
        bench_err_sq = np.sum((y_real - daily_pm_bench)**2)

        oos_num_raw.append(err_sq_raw)
        oos_denominator_list.append(bench_err_sq)

        performance_log.append({
            'Date': date, 'IC_Raw': ic_raw,
            'R2_OS_Raw': 1 - (err_sq_raw / (bench_err_sq + 1e-8)),
            'Turnover': np.sum(np.abs(weights - curr_last_w))
        })

# --- 4. 导出 ---
if weight_records:
    total_bench = sum(oos_denominator_list) + 1e-8
    final_r2_raw = 1 - (sum(oos_num_raw) / total_bench)
    perf_df = pd.DataFrame(performance_log)
    print(f"\n✅ LightGBM 汇总 R2_OS: {final_r2_raw:.6f} | 平均 IC: {perf_df['IC_Raw'].mean():.4f}")
    
    perf_df.to_excel('LGBM_Monthly_Diagnostics.xlsx', index=False)
    pd.concat(weight_records)[lambda x: x['weight'] > 0].to_excel('LGBM_Monthly_Weights.xlsx', index=False)

🚀 启动 Huber-LGBM 滚动回测 (固定参数 + 全量训练 + 单股上限 1.0)...


100%|██████████| 514/514 [02:21<00:00,  3.64it/s]



✅ LightGBM 汇总 R2_OS: 0.051730 | 平均 IC: 0.0116


In [8]:
import pandas as pd
import numpy as np
import cvxpy as cp
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.covariance import LedoitWolf
from tqdm import tqdm
import scipy.stats as stats
import warnings
import gc

warnings.filterwarnings("ignore")

# 辅助函数
def numpy_spearmanr(x, y):
    if len(x) < 2: return 0.0
    res = stats.spearmanr(x, y)
    return res.correlation if hasattr(res, 'correlation') else res[0]

# --- 1. 配置与数据准备 ---
df = df_factor_monthly.copy()
target = 'target_monthly_ret_y' 
features = [col for col in df.columns if col.startswith('factor_')]

oos_start, oos_end = '2013-06-30', '2024-12-31'
lookback_window = 60    # 统一 60 个月回溯
train_step = 6         # 每半年重新训练底层子模型
risk_window_len = 36    

df['TradingDate'] = pd.to_datetime(df['TradingDate'])
all_dates = sorted(df['TradingDate'].unique())
rebalance_dates = [d for d in all_dates if pd.to_datetime(oos_start) <= d <= pd.to_datetime(oos_end)]
df_lookup = {d: group for d, group in df.groupby('TradingDate')}

# --- 2. 底层模型固定参数配置 ---

# Ridge 固定参数
RIDGE_ALPHA = 5000

# LightGBM 固定参数
lgbm_params = {
    'n_estimators': 80,           # 降低迭代次数
    'learning_rate': 0.005,       # 极致低学习率，增加收敛稳定性
    'max_depth': 2,               # 【核心】降到 2 层。强制捕捉最简单的逻辑
    'num_leaves': 3,              # 配合 max_depth=2，叶子数只能是 3 或 4
    'subsample': 0.7,             # 样本采样，增加鲁棒性
    'subsample_freq': 1,
    'colsample_bytree': 0.6,      # 特征采样，防止模型依赖某几个特定因子
    'lambda_l1': 10.0,            # 增加 L1 正则，让模型自动剔除弱因子
    'lambda_l2': 500.0,           # 【大幅提升】L2 正则，压低系数绝对值
    'min_child_samples': 50,      # 强制每个叶子节点必须有更多样本，防止对个别股票过拟合
    'objective': 'huber',         # 维持 Huber，对抗 A 股收益率的厚尾噪声
    'verbosity': -1,
    'random_state': 2026,
    'n_jobs': -1
}

# XGBoost 固定参数
xgb_params = {
    'n_estimators': 80,           # 适度减少树的数量，防止在长窗口末端过拟合
    'max_depth': 2,               # 【关键】从 3 降到 2。深度为 2 的树只能捕捉二阶交互，更稳健
    'learning_rate': 0.005,       # 进一步降低学习率，让模型收敛更慢、更平滑
    'subsample': 0.7,             # 引入行采样，增加基学习器之间的差异性
    'colsample_bytree': 0.6,      # 降低列采样比例，强制模型在训练每棵树时只看部分因子
    'reg_lambda': 500.0,          # 【大幅提升】L2 正则化从 100 提到 500，强行压低叶子节点权重
    'reg_alpha': 10.0,            # 引入 L1 正则化，让一些贡献极小的无效因子权重彻底归零
    'objective': 'reg:squarederror', # 尝试换回平方损失，配合高正则化，通常对解释 $R^2$ 更客观
    'tree_method': 'hist',
    'n_jobs': -1,
    'random_state': 2026,
    'verbosity': 0
}

# --- 3. 状态存储变量 ---
weight_records, performance_log = [], [] 
oos_num_raw, oos_denominator_list = [], [] 
last_w_val = None 
models = {} 

print(f"🚀 启动 EW-Ensemble 策略 (底层全部固定参数对齐版)...")

# --- 4. 滚动回测循环 ---
for i in tqdm(range(len(rebalance_dates))):
    date = rebalance_dates[i]
    date_idx = all_dates.index(date)
    if date_idx < lookback_window: continue
    
    # --- 4.1 动态再训练底层子模型 (固定参数 + 全量训练) ---
    if i % train_step == 0 or not models:
        gc.collect()
        train_dates = all_dates[date_idx - lookback_window : date_idx]
        full_train = pd.concat([df_lookup[d] for d in train_dates if d in df_lookup])[[target] + features].dropna()
        
        X_train_all = full_train[features]
        y_train_all = full_train[target]
        
        # A. 训练 Ridge (固定 Alpha)
        m_ridge = Ridge(alpha=RIDGE_ALPHA).fit(X_train_all, y_train_all)
        
        # B. 训练 LightGBM (固定迭代次数)
        m_lgb = lgb.LGBMRegressor(**lgbm_params)
        m_lgb.fit(X_train_all, y_train_all)
        
        # C. 训练 XGBoost (固定迭代次数)
        m_xgb = xgb.XGBRegressor(**xgb_params)
        m_xgb.fit(X_train_all, y_train_all)
        
        models = {'rid': m_ridge, 'xgb': m_xgb, 'lgb': m_lgb}
        del full_train; gc.collect()

    # --- 4.2 信号合成 (1/3 等权集成) ---
    current_day = df_lookup.get(date)
    if current_day is None: continue
    current_day = current_day.dropna(subset=features).copy()
    
    p_rid = models['rid'].predict(current_day[features])
    p_xgb = models['xgb'].predict(current_day[features])
    p_lgb = models['lgb'].predict(current_day[features])
    
    mu_raw_output = (p_rid + p_xgb + p_lgb) / 3.0
    mu_series = pd.Series(mu_raw_output, index=current_day['Symbol'].tolist())

    # --- 4.3 MVO 资产配置 ---
    trade_candidates = current_day[current_day['is_hs300'] == True]['Symbol'].tolist()
    risk_dates = all_dates[date_idx - risk_window_len : date_idx]
    pm_dates = all_dates[max(0, date_idx - 60) : date_idx]
    
    risk_hist_df = pd.concat([df_lookup[d][['TradingDate', 'Symbol', target]] for d in risk_dates if d in df_lookup])
    
    v_matrix = risk_hist_df[risk_hist_df['Symbol'].isin(trade_candidates)].pivot(index='TradingDate', columns='Symbol', values=target).fillna(0)
    active_symbols = [s for s in trade_candidates if s in v_matrix.columns]
    
    if len(active_symbols) < 50: continue 

    mu_for_mvo = mu_series.reindex(active_symbols).fillna(0).values
    Sigma = LedoitWolf().fit(v_matrix[active_symbols].values).covariance_
    Sigma = (Sigma + Sigma.T) / 2
    
    n = len(active_symbols)
    w = cp.Variable(n)
    curr_last_w = last_w_val.reindex(active_symbols).fillna(0).values if last_w_val is not None else np.ones(n)/n
    
    # 目标函数
    obj = cp.Maximize(mu_for_mvo @ w - 0.5 * 5.0 * cp.quad_form(w, Sigma) - 0.03 * cp.norm(w - curr_last_w, 1))
    constraints = [cp.sum(w) == 1, w >= 0, w <= 1.0] 
    
    try:
        prob = cp.Problem(obj, constraints)
        prob.solve(solver=cp.OSQP)
        weights = np.array(w.value).flatten()
        if weights is None: raise ValueError()
        weights[weights < 1e-4] = 0; weights /= weights.sum()
    except:
        weights = curr_last_w
        
    last_w_val = pd.Series(weights, index=active_symbols)
    weight_records.append(pd.DataFrame({'date': date, 'symbol': active_symbols, 'weight': weights}))
    
    # --- 4.4 统计诊断 ---
    if target in current_day.columns:
        y_real = current_day.set_index('Symbol').reindex(active_symbols)[target].values
        pm_hist_df_bench = pd.concat([df_lookup[d][['Symbol', target]] for d in pm_dates if d in df_lookup])
        daily_pm_bench = pm_hist_df_bench.groupby('Symbol')[target].mean().reindex(active_symbols).fillna(0).values

        err_sq_raw = np.sum((y_real - mu_for_mvo)**2)
        ic_raw = numpy_spearmanr(mu_for_mvo, y_real)
        bench_err_sq = np.sum((y_real - daily_pm_bench)**2)

        oos_num_raw.append(err_sq_raw)
        oos_denominator_list.append(bench_err_sq)

        performance_log.append({
            'Date': date, 'IC_Raw': ic_raw,
            'R2_OS_Raw': 1 - (err_sq_raw / (bench_err_sq + 1e-8)),
            'Turnover': np.sum(np.abs(weights - curr_last_w))
        })

# --- 5. 汇总导出 ---
if weight_records:
    total_bench = sum(oos_denominator_list) + 1e-8
    final_r2_raw = 1 - (sum(oos_num_raw) / total_bench)
    perf_df = pd.DataFrame(performance_log)
    
    print("\n" + "="*45)
    print(f"📊 EW-Ensemble 汇总 R2_OS: {final_r2_raw:.6f} | 平均 IC: {perf_df['IC_Raw'].mean():.4f}")
    print("="*45)
    
    perf_df.to_excel('Monthly_EW_Ensemble_Diagnostics.xlsx', index=False)
    pd.concat(weight_records)[lambda x: x['weight'] > 0].to_excel('Monthly_EW_Ensemble_Weights.xlsx', index=False)

🚀 启动 EW-Ensemble 策略 (底层全部固定参数对齐版)...


100%|██████████| 514/514 [02:31<00:00,  3.38it/s]



📊 EW-Ensemble 汇总 R2_OS: 0.050191 | 平均 IC: 0.0344


In [6]:
import pandas as pd
import numpy as np
import cvxpy as cp
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge, ElasticNetCV
from sklearn.covariance import LedoitWolf
from tqdm import tqdm
import scipy.stats as stats
import warnings
import gc

warnings.filterwarnings("ignore")

# 辅助函数
def numpy_spearmanr(x, y):
    if len(x) < 2: return 0.0
    res = stats.spearmanr(x, y)
    return res.correlation if hasattr(res, 'correlation') else res[0]

# --- 1. 配置与数据准备 ---
df = df_factor_monthly.copy()
target = 'target_monthly_ret_y' 
features = [col for col in df.columns if col.startswith('factor_')]

oos_start, oos_end = '2013-06-30', '2024-12-31'
lookback_window = 60    # 统一 60 个月窗口
train_step = 6         # 统一半年更新一次
risk_window_len = 36    

df['TradingDate'] = pd.to_datetime(df['TradingDate'])
all_dates = sorted(df['TradingDate'].unique())
rebalance_dates = [d for d in all_dates if pd.to_datetime(oos_start) <= d <= pd.to_datetime(oos_end)]
df_lookup = {d: group for d, group in df.groupby('TradingDate')}

# --- 2. 底层子模型固定参数 (与单模型严格对齐) ---
RIDGE_ALPHA = 5000

# LightGBM 固定参数
lgbm_params = {
    'n_estimators': 80,           # 降低迭代次数
    'learning_rate': 0.005,       # 极致低学习率，增加收敛稳定性
    'max_depth': 2,               # 【核心】降到 2 层。强制捕捉最简单的逻辑
    'num_leaves': 3,              # 配合 max_depth=2，叶子数只能是 3 或 4
    'subsample': 0.7,             # 样本采样，增加鲁棒性
    'subsample_freq': 1,
    'colsample_bytree': 0.6,      # 特征采样，防止模型依赖某几个特定因子
    'lambda_l1': 10.0,            # 增加 L1 正则，让模型自动剔除弱因子
    'lambda_l2': 500.0,           # 【大幅提升】L2 正则，压低系数绝对值
    'min_child_samples': 50,      # 强制每个叶子节点必须有更多样本，防止对个别股票过拟合
    'objective': 'huber',         # 维持 Huber，对抗 A 股收益率的厚尾噪声
    'verbosity': -1,
    'random_state': 2026,
    'n_jobs': -1
}

# XGBoost 固定参数
xgb_params = {
    'n_estimators': 80,           # 适度减少树的数量，防止在长窗口末端过拟合
    'max_depth': 2,               # 【关键】从 3 降到 2。深度为 2 的树只能捕捉二阶交互，更稳健
    'learning_rate': 0.005,       # 进一步降低学习率，让模型收敛更慢、更平滑
    'subsample': 0.7,             # 引入行采样，增加基学习器之间的差异性
    'colsample_bytree': 0.6,      # 降低列采样比例，强制模型在训练每棵树时只看部分因子
    'reg_lambda': 500.0,          # 【大幅提升】L2 正则化从 100 提到 500，强行压低叶子节点权重
    'reg_alpha': 10.0,            # 引入 L1 正则化，让一些贡献极小的无效因子权重彻底归零
    'objective': 'reg:squarederror', # 尝试换回平方损失，配合高正则化，通常对解释 $R^2$ 更客观
    'tree_method': 'hist',
    'n_jobs': -1,
    'random_state': 2026,
    'verbosity': 0
}

# --- 3. 状态存储变量 ---
weight_records, performance_log = [], [] 
oos_num_raw, oos_denominator_list = [], [] 
last_w_val, last_ensemble_weights = None, None 

print(f"🚀 启动 C-ENet 动态集成策略 (底层固定参数 + 全量训练版)...")

# --- 4. 滚动预测与优化循环 ---
for i in tqdm(range(len(rebalance_dates))):
    date = rebalance_dates[i]
    date_idx = all_dates.index(date)
    if date_idx < lookback_window: continue
    
    # --- 4.1 动态再训练底层模型及集成权重 ---
    if i % train_step == 0 or 'model_lgb' not in locals():
        gc.collect()
        train_dates = all_dates[date_idx - lookback_window : date_idx]
        full_train = pd.concat([df_lookup[d] for d in train_dates if d in df_lookup])[[target] + features].dropna()
        
        # 直接使用全量数据训练子模型，不再拆分验证集
        X_all = full_train[features]
        y_all = full_train[target]
        
        # A. 训练子模型 1: Ridge
        model_ridge = Ridge(alpha=RIDGE_ALPHA).fit(X_all, y_all)
        
        # B. 训练子模型 2: LightGBM
        model_lgb = lgb.LGBMRegressor(**lgbm_params)
        model_lgb.fit(X_all, y_all)
        
        # C. 训练子模型 3: XGBoost
        model_xgb = xgb.XGBRegressor(**xgb_params)
        model_xgb.fit(X_all, y_all)
        
        # D. C-ENet 集成权重计算逻辑 (核心逻辑不变)
        # 生成过去窗口内子模型的预测值作为 ElasticNet 的输入
        comb_list = []
        for d in train_dates:
            if d in df_lookup:
                day_data = df_lookup[d].dropna(subset=features)
                if day_data.empty: continue
                comb_list.append(pd.DataFrame({
                    'lgb': model_lgb.predict(day_data[features]),
                    'xgb': model_xgb.predict(day_data[features]),
                    'rid': model_ridge.predict(day_data[features]),
                    'target': day_data[target].values
                }))
        
        c_df = pd.concat(comb_list).dropna()
        # 使用 ElasticNet 决定谁的预测更靠谱
        enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], alphas=[1e-5, 1e-4, 1e-3, 0.01, 0.1], 
                            positive=True, fit_intercept=False, cv=5)
        enet.fit(c_df[['lgb', 'xgb', 'rid']], c_df['target'])
        
        # 权重归一化与平滑
        sum_coef = enet.coef_.sum()
        new_raw_weights = enet.coef_ / sum_coef if sum_coef > 1e-6 else np.array([1/3, 1/3, 1/3])
        
        if last_ensemble_weights is None:
            current_ensemble_weights = new_raw_weights
        else:
            # 引入 0.4 的平滑系数，防止集成权重变动过大
            current_ensemble_weights = 0.9 * last_ensemble_weights + 0.1 * new_raw_weights
        last_ensemble_weights = current_ensemble_weights
        
        del full_train; gc.collect()

    # --- 4.2 信号合成 ---
    current_day = df_lookup.get(date)
    if current_day is None: continue
    current_day = current_day.dropna(subset=features).copy()
    
    raw_preds_matrix = np.column_stack([
        model_lgb.predict(current_day[features]),
        model_xgb.predict(current_day[features]),
        model_ridge.predict(current_day[features])
    ])
    
    mu_raw_output = raw_preds_matrix @ current_ensemble_weights
    mu_series = pd.Series(mu_raw_output, index=current_day['Symbol'].tolist())

    # --- 4.3 MVO 资产配置 ---
    trade_candidates = current_day[current_day['is_hs300'] == True]['Symbol'].tolist()
    risk_dates = all_dates[date_idx - risk_window_len : date_idx]
    pm_dates = all_dates[max(0, date_idx - 60) : date_idx]
    
    v_matrix = pd.concat([df_lookup[d][['TradingDate', 'Symbol', target]] for d in risk_dates if d in df_lookup])\
               .query("Symbol in @trade_candidates")\
               .pivot(index='TradingDate', columns='Symbol', values=target).fillna(0)
    active_symbols = v_matrix.columns.tolist()
    
    if len(active_symbols) < 50: continue 

    mu_for_mvo = mu_series.reindex(active_symbols).fillna(0).values
    Sigma = LedoitWolf().fit(v_matrix.values).covariance_
    Sigma = (Sigma + Sigma.T) / 2
    
    n = len(active_symbols)
    w = cp.Variable(n)
    curr_last_w = last_w_val.reindex(active_symbols).fillna(0).values if last_w_val is not None else np.ones(n)/n
    
    obj = cp.Maximize(mu_for_mvo @ w - 0.5 * 5.0 * cp.quad_form(w, Sigma) - 0.03 * cp.norm(w - curr_last_w, 1))
    constraints = [cp.sum(w) == 1, w >= 0, w <= 1.0] 
    
    try:
        prob = cp.Problem(obj, constraints); prob.solve(solver=cp.OSQP)
        weights = np.array(w.value).flatten()
        if weights is None: raise ValueError()
        weights[weights < 1e-4] = 0; weights /= weights.sum()
    except:
        weights = curr_last_w
        
    last_w_val = pd.Series(weights, index=active_symbols)
    weight_records.append(pd.DataFrame({'date': date, 'symbol': active_symbols, 'weight': weights}))
    
    # --- 4.4 统计诊断 ---
    if target in current_day.columns:
        y_real = current_day.set_index('Symbol').reindex(active_symbols)[target].values
        pm_hist_df_bench = pd.concat([df_lookup[d][['Symbol', target]] for d in pm_dates if d in df_lookup])
        daily_pm_bench = pm_hist_df_bench.groupby('Symbol')[target].mean().reindex(active_symbols).fillna(0).values

        err_sq_raw = np.sum((y_real - mu_for_mvo)**2)
        ic_raw = numpy_spearmanr(mu_for_mvo, y_real)
        bench_err_sq = np.sum((y_real - daily_pm_bench)**2)
        oos_num_raw.append(err_sq_raw); oos_denominator_list.append(bench_err_sq)

        performance_log.append({
            'Date': date, 'IC_Raw': ic_raw,
            'R2_OS_Raw': 1 - (err_sq_raw / (bench_err_sq + 1e-8)),
            'Turnover': np.sum(np.abs(weights - curr_last_w)),
            'W_LGB': current_ensemble_weights[0], 
            'W_XGB': current_ensemble_weights[1], 
            'W_RID': current_ensemble_weights[2]
        })

# --- 5. 导出 ---
if weight_records:
    perf_df = pd.DataFrame(performance_log)
    total_bench = sum(oos_denominator_list) + 1e-8
    final_r2_raw = 1 - (sum(oos_num_raw) / total_bench)
    print(f"\n✅ C-ENet 汇总 R2_OS: {final_r2_raw:.6f} | 平均 IC: {perf_df['IC_Raw'].mean():.4f}")
    
    perf_df.to_excel('CENet_Monthly_Diagnostics.xlsx', index=False)
    pd.concat(weight_records)[lambda x: x['weight'] > 0].to_excel('HS300_CENet_Monthly_Final_Weights.xlsx', index=False)

🚀 启动 C-ENet 动态集成策略 (底层固定参数 + 全量训练版)...


100%|██████████| 514/514 [03:58<00:00,  2.16it/s]



✅ C-ENet 汇总 R2_OS: 0.048617 | 平均 IC: 0.0368


In [69]:
import pandas as pd
import numpy as np
import cvxpy as cp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.covariance import LedoitWolf
from scipy.stats import spearmanr
from tqdm import tqdm
import warnings
import gc
import random

warnings.filterwarnings("ignore")

# 锁定随机种子确保结果可复现
def set_seed(seed=2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(2026)

def numpy_spearmanr(x, y):
    if len(x) < 2: return 0.0
    res = spearmanr(x, y)
    return res.correlation if hasattr(res, 'correlation') else res[0]

# --- 0. AI 模型定义 ---
class LSTM_VSN_Model(nn.Module):
    def __init__(self, input_dim, hidden_dim=4):
        super(LSTM_VSN_Model, self).__init__()
        # VSN Lite: 简单的变量选择网络思想
        self.vsn_lite = nn.Linear(input_dim, input_dim) 
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        x = torch.relu(self.vsn_lite(x))
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

# --- 1. 配置与数据准备 ---
df = df_factor_monthly.copy()
target = 'target_monthly_ret_y' 
features = [col for col in df.columns if col.startswith('factor_')]

oos_start, oos_end = '2013-06-30', '2024-12-31'
lookback_window = 60    # 5年回溯
train_step = 6         # 半年更新一次
risk_window_len = 36    # 统一 36 个月风险窗口

# 固定深度学习超参数
HIDDEN_DIM = 4         
FIXED_EPOCHS = 30       # 固定训练轮数，取消早停
BATCH_SIZE = 256       
LEARNING_RATE = 0.001
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df['TradingDate'] = pd.to_datetime(df['TradingDate'])
all_dates = sorted(df['TradingDate'].unique())
rebalance_dates = [d for d in all_dates if pd.to_datetime(oos_start) <= d <= pd.to_datetime(oos_end)]
df_lookup = {d: group for d, group in df.groupby('TradingDate')}

# --- 2. 状态存储 ---
weight_records, performance_log = [], [] 
oos_num_raw, oos_denominator_list = [], [] 
last_w_val, model = None, None

print(f"🚀 启动固定参数+全量训练版 LSTM-VSN 策略 (固定 Epochs={FIXED_EPOCHS} + 原始量级预测)...")

# --- 3. 滚动预测与优化 ---
for i in tqdm(range(len(rebalance_dates))):
    date = rebalance_dates[i]
    date_idx = all_dates.index(date)
    if date_idx < lookback_window: continue
    
    # --- 3.1 动态再训练 (固定 Epochs，取消验证集) ---
    if i % train_step == 0 or model is None:
        gc.collect(); torch.cuda.empty_cache()
        train_dates = all_dates[date_idx - lookback_window : date_idx]
        full_train_df = pd.concat([df_lookup[d] for d in train_dates if d in df_lookup])[[target] + features].dropna()
        
        # 准备全量训练数据
        X_train_np = full_train_df[features].values
        # 收益率去极值处理，保持标签数值稳定性
        y_train_np = np.clip(full_train_df[target].values, -0.2, 0.2)
        
        X_t = torch.from_numpy(X_train_np).float().unsqueeze(1).to(device)
        y_t = torch.from_numpy(y_train_np).float().view(-1, 1).to(device)
        
        # 重新初始化模型
        model = LSTM_VSN_Model(len(features), HIDDEN_DIM).to(device)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.HuberLoss(delta=1.0) # 使用 HuberLoss 增强稳健性
        
        train_loader = DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE, shuffle=True)
        
        model.train()
        for epoch in range(FIXED_EPOCHS):
            for bx, by in train_loader:
                optimizer.zero_grad()
                outputs = model(bx)
                loss = criterion(outputs, by)
                loss.backward()
                optimizer.step()
        
        del X_t, y_t, full_train_df; gc.collect()

    # --- 3.2 样本外原始预测 (已移除 Z-Score 标准化) ---
    current_day = df_lookup.get(date)
    if current_day is None: continue
    current_day = current_day.dropna(subset=features).copy()
    
    model.eval()
    with torch.no_grad():
        X_oos = torch.from_numpy(current_day[features].values).float().unsqueeze(1).to(device)
        mu_raw = model(X_oos).cpu().numpy().flatten()
    
    # 直接使用模型生成的原始信号 mu_raw
    mu_series = pd.Series(mu_raw, index=current_day['Symbol'].tolist())

    # --- 3.3 MVO 资产配置 ---
    trade_candidates = current_day[current_day['is_hs300'] == True]['Symbol'].tolist()
    risk_dates = all_dates[date_idx - risk_window_len : date_idx]
    pm_dates = all_dates[max(0, date_idx - 36) : date_idx]
    
    risk_hist_df = pd.concat([df_lookup[d][['TradingDate', 'Symbol', target]] for d in risk_dates if d in df_lookup])
    pm_hist_df = pd.concat([df_lookup[d][['Symbol', target]] for d in pm_dates if d in df_lookup])
    rolling_pm_series = pm_hist_df.groupby('Symbol')[target].mean()
    
    v_matrix = risk_hist_df[risk_hist_df['Symbol'].isin(trade_candidates)].pivot(index='TradingDate', columns='Symbol', values=target).fillna(0)
    active_symbols = [s for s in trade_candidates if s in v_matrix.columns]
    
    if len(active_symbols) < 50: continue 

    # 获取原始量级的 mu
    mu_for_mvo = mu_series.reindex(active_symbols).fillna(0).values
    
    # 协方差收缩处理 (Ledoit-Wolf)
    Sigma = LedoitWolf().fit(v_matrix[active_symbols].values).covariance_
    Sigma = 0.8 * Sigma + 0.2 * np.diag(np.diag(Sigma)) # 增加对角线权重降低虚假相关性噪声
    Sigma = (Sigma + Sigma.T) / 2
    
    n = len(active_symbols)
    w = cp.Variable(n)
    curr_w = last_w_val.reindex(active_symbols).fillna(0).values if last_w_val is not None else np.ones(n)/n
    
    # 目标函数：γ=5，换手惩罚=0.03
    obj = cp.Maximize(mu_for_mvo @ w - 0.5 * 5.0 * cp.quad_form(w, Sigma) - 0.03 * cp.norm(w - curr_w, 1))
    # 强制分散约束：单股上限 5%
    constraints = [cp.sum(w) == 1, w >= 0, w <= 1] 
    
    try:
        prob = cp.Problem(obj, constraints)
        prob.solve(solver=cp.OSQP, eps_abs=1e-4, eps_rel=1e-4)
        weights = np.array(w.value).flatten()
        if weights is None: raise ValueError()
        weights[weights < 1e-4] = 0; weights /= weights.sum()
    except:
        weights = curr_w
        
    last_w_val = pd.Series(weights, index=active_symbols)
    weight_records.append(pd.DataFrame({'date': date, 'symbol': active_symbols, 'weight': weights}))
    
    # --- 3.4 统计诊断 ---
    if target in current_day.columns:
        y_real = current_day.set_index('Symbol').reindex(active_symbols)[target].values
        daily_pm_bench = rolling_pm_series.reindex(active_symbols).fillna(0).values
        
        err_sq_raw = np.sum((y_real - mu_for_mvo)**2)
        ic_raw = numpy_spearmanr(mu_for_mvo, y_real)
        bench_err_sq = np.sum((y_real - daily_pm_bench)**2)

        oos_num_raw.append(err_sq_raw)
        oos_denominator_list.append(bench_err_sq)

        performance_log.append({
            'Date': date, 
            'IC_Raw': ic_raw,
            'R2_OS_Raw': 1 - (err_sq_raw / (bench_err_sq + 1e-8)),
            'Turnover': np.sum(np.abs(weights - curr_w))
        })

# --- 4. 统计导出 ---
if weight_records:
    total_bench = sum(oos_denominator_list) + 1e-8
    final_r2_raw = 1 - (sum(oos_num_raw) / total_bench)
    perf_df = pd.DataFrame(performance_log)
    print("\n" + "="*45)
    print(f"📊 原始量级版 LSTM-VSN-MVO 诊断完成:")
    print(f"全样本 $R^2_{{OS}}$: {final_r2_raw:.6f} | 平均 IC: {perf_df['IC_Raw'].mean():.4f}")
    print("="*45)
    
    perf_df.to_excel('LSTM_VSN_Monthly_Diagnostics.xlsx', index=False)
    pd.concat(weight_records)[lambda x: x['weight'] > 0].to_excel('LSTM_VSN_Monthly_Weights.xlsx', index=False)

🚀 启动固定参数+全量训练版 LSTM-VSN 策略 (固定 Epochs=30 + 原始量级预测)...


100%|██████████| 514/514 [07:37<00:00,  1.12it/s]


📊 原始量级版 LSTM-VSN-MVO 诊断完成:
全样本 $R^2_{OS}$: -0.080235 | 平均 IC: 0.0129
